In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
from time import sleep
import re

# 

In [3]:
HEADERS = {
    'User-Agent': 'qangos-scraper/1.0(contact:qangos.bsdsba2028@aim.edu)'
}


def request_site(
    url: str,
    headers: dict = HEADERS,
    sleep_time: int = 1,
    ignore_message: bool = False,
    params: dict = None
):
    """
    Execute GET request on specified url. Returns response
    """
    try:
        if not params:
            response = requests.get(url, headers=headers)
        else:
            response = requests.get(url, headers=headers, params=params)
        sleep(sleep_time)
        response.raise_for_status()
    except requests.RequestException as e:
        print("Failed to fetch page:", e)
    else:
        if not ignore_message:
            print(f"Page fetched! {url}")
    return response

In [4]:
ref_regex = re.compile(r'\[\d+\]')

def clean_item(item):
    return ref_regex.sub('', item)

def get_item(item, tag='title'):
    if not item.find('a'):
        if tag=='href':
            return None
        return clean_item(item.text.strip())

    if tag.lower() == 'href':
        if item.a['href'].startswith('#'):
            return None
        if not item.a['href'].startswith('/'):
            return None
        return clean_item(item.a['href'])

    if item.a.has_attr(tag):
        return clean_item(item.a['title'])

    return clean_item(item.text.strip())


def get_game_data(row):
    items = row.find_all('td')

    game = get_item(items[0])
    link = get_item(items[0], 'href')
    publisher = get_item(items[1])
    system = get_item(items[2])
    release = get_item(items[3])
    setting = get_item(items[4])

    return {
        'game': game,
        'link': link,
        'publisher': publisher,
        'system': system,
        'release_year': release,
        'setting': setting
    }

In [188]:
link = 'https://en.wikipedia.org/wiki/List_of_tabletop_role-playing_games'
page = request_site(link)
soup = BeautifulSoup(page.content)

Page fetched! https://en.wikipedia.org/wiki/List_of_tabletop_role-playing_games


In [196]:
games = []
for i in soup.tbody.find_all('tr')[1:]:
    # check if row info is complete, skip if not
    if len(i.find_all('td')) != 6:
        continue

    games.append(get_game_data(i))

In [199]:
games_df = pd.DataFrame(games)
games_df.head()

,game,link,publisher,system,release_year,setting
0,13th Age,/wiki/13th_Age,Pelgrane Press,D20 System,2013,High fantasy
1,The 23rd Letter,/wiki/The_23rd_Letter,Crucible Design,,1996,
2,2300 AD,/wiki/2300_AD,Game Designers' Workshop,,1989,Hard science fiction
3,3D&T,/wiki/3D%26T,Editora Talismã (formerly Trama Editorial),,1994,
4,7th Sea (role-playing game),/wiki/7th_Sea_(role-playing_game),Alderac Entertainment Group,D20 System,"1999, 2016",Pirate


In [244]:
def get_game_description(link):
    if not link:
        return None
        
    l = f'https://en.wikipedia.org/{link}'
    page = request_site(l)
    soup = BeautifulSoup(page.content)
    
    return clean_item(' '.join([p.text.strip() for p in soup.find_all('p')])).strip()

In [245]:
games_df['description'] = games_df['link'].apply(get_game_description)

Page fetched! https://en.wikipedia.org//wiki/13th_Age
Page fetched! https://en.wikipedia.org//wiki/The_23rd_Letter
Page fetched! https://en.wikipedia.org//wiki/2300_AD
Page fetched! https://en.wikipedia.org//wiki/3D%26T
Page fetched! https://en.wikipedia.org//wiki/7th_Sea_(role-playing_game)
Page fetched! https://en.wikipedia.org//wiki/Aberrant_(role-playing_game)
Page fetched! https://en.wikipedia.org//wiki/ACE_Agents!
Page fetched! https://en.wikipedia.org//wiki/Active_Exploits
Page fetched! https://en.wikipedia.org//wiki/Editions_of_Dungeons_%26_Dragons#Advanced_Dungeons_&_Dragons
Page fetched! https://en.wikipedia.org//wiki/Advanced_Fighting_Fantasy
Page fetched! https://en.wikipedia.org//wiki/Adventure!_(role-playing_game)
Page fetched! https://en.wikipedia.org//wiki/Adventure!
Page fetched! https://en.wikipedia.org//wiki/Adventurers_Guild
Page fetched! https://en.wikipedia.org//wiki/Adventures_in_Fantasy
Page fetched! https://en.wikipedia.org//wiki/The_Adventures_of_Indiana_Jones

In [254]:
games_df[~games_df['description'].isna()].to_csv('test_scrape.csv', index=False)

In [248]:
games_df

,game,link,publisher,system,release_year,setting,description
0,13th Age,/wiki/13th_Age,Pelgrane Press,D20 System,2013,High fantasy,13th Age is a d20 fantasy role-playing game de...
1,The 23rd Letter,/wiki/The_23rd_Letter,Crucible Design,,1996,,This is a list of notable tabletop role-playin...
2,2300 AD,/wiki/2300_AD,Game Designers' Workshop,,1989,Hard science fiction,"The tabletop game 2300 AD, originally titled T..."
3,3D&T,/wiki/3D%26T,Editora Talismã (formerly Trama Editorial),,1994,,"3DeT, formerly known as 3D&T or Defensores de ..."
4,7th Sea (role-playing game),/wiki/7th_Sea_(role-playing_game),Alderac Entertainment Group,D20 System,"1999, 2016",Pirate,"7th Sea is a ""swashbuckling and sorcery""-theme..."
...,...,...,...,...,...,...,...
782,Year of the Phoenix,/wiki/Year_of_the_Phoenix,Fantasy Games Unlimited,,1986,,Year of the Phoenix is a science-fiction role-...
783,Ysgarth,/wiki/Ysgarth,Ragnarok Games,,1979,,Ysgarth is a fantasy role-playing game publish...
784,The Zorcerer of Zo,/wiki/The_Zorcerer_of_Zo,Atomic Sock Monkey Press,,2006,Fairy tales,The Zantabulous Zorcerer of Zo is a fairy tale...
785,Zweihänder (role-playing game),/wiki/Zweih%C3%A4nder_(role-playing_game),Grim & Perilous Studios,,2017,,Zweihänder Grim & Perilous RPG or Zweihänder i...
